# 🧠 ACA N° 2 – Aprendizaje Automático
## Redes neuronales para predecir la deserción universitaria · Proyecto 2: Educación
**Dataset:** *Predict Students' Dropout and Academic Success* (UCI 697) · **Modelos:** MLP con *embeddings* y FT-Transformer (PyTorch) · **Despliegue:** Streamlit

Este cuaderno continúa el ACA 1 (mismo dataset, misma limpieza, misma partición 80/20 y misma validación cruzada) y reemplaza el modelado clásico por **redes neuronales**, siguiendo los 5 pasos del enunciado:

| Paso | Parte del cuaderno |
|---|---|
| 1 · Exploración y gestión **SQL** | Parte 1 |
| 2 · Preparación y pipeline | Parte 2 |
| 3 · Modelado neuronal | Partes 3, 4 y 5 |
| 4 · Evaluación y métricas (loss, accuracy, F1, matriz de confusión, curvas) | Partes 6 y 7 |
| 5 · Despliegue en Streamlit | Parte 8 (exportación) y `app.py` |

**Pregunta orientadora.** ¿Cómo aprovechar arquitecturas de redes neuronales para extraer patrones complejos en datos educativos, optimizando el rendimiento predictivo y permitiendo su uso práctico en una interfaz Streamlit?

> ⚙️ **Cómo ejecutarlo.** Funciona en local o en Google Colab (CPU es suficiente). Ejecute *Run all*. Con `QUICK = False` tarda unos 15–30 min en CPU; con `QUICK = True` (prueba rápida) unos 5 min. Al final genera `models/*.pt` y `models/*_nn*.json`, que usa la app.

> 📝 **Borrador sin ejecutar.** Este cuaderno se entrega listo para correr pero **sin salidas**: requiere PyTorch, que no se instaló en el entorno donde se escribió. Ejecútelo en **Google Colab (CPU, gratis; PyTorch ya viene instalado)** con *Run all*. Si algo falla, copie el error y se corrige. Al terminar genera `models/*.pt` y `models/*_nn*.json`, que la app necesita. Para una prueba rápida de ~5 min ponga `QUICK = True` en la celda de configuración.
>
> **En Colab:** abra el notebook, ejecute la primera celda de código y se clonará automáticamente el repo público (`REPO_URL`). Debe estar subido el proyecto completo (`src/`, `data/`, etc.).

## 0. Configuración del entorno

In [ ]:
import sys, os, json, time, random, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# --- Ubicar la raíz del repositorio (local o Google Colab) -------------------------------------
# En Colab el notebook suelto NO trae la carpeta src/. Opciones (el cuaderno prueba en este orden):
#   1) la carpeta ya está en /content (subida o descomprimida);  2) está en Drive (/content/drive/MyDrive/aca_2_machine_learning);
#   3) REPO_URL con la URL de GitHub;  4) si nada de lo anterior, le pedirá subir el .zip del proyecto.
REPO_URL = "https://github.com/Paula217/edupredict-nn-aca2.git"   # repo público del ACA 2 (si no se encuentra la carpeta en Colab, se clona de aquí)
IN_COLAB = "google.colab" in sys.modules

def _es_raiz(p: Path) -> bool:
    return (p / "src" / "data_loader.py").exists()

def _buscar_raiz():
    cwd = Path.cwd()
    candidatas = [cwd, cwd.parent, Path("/content/aca_2_machine_learning"), Path("/content/repo"),
                  Path("/content/drive/MyDrive/aca_2_machine_learning")]
    for c in candidatas:
        if _es_raiz(c): return c
    for c in Path("/content").glob("*/") if Path("/content").exists() else []:      # una carpeta anidada en /content
        for sub in [c] + list(c.glob("*/")):
            if _es_raiz(sub): return sub
    return None

ROOT = _buscar_raiz()
if ROOT is None and IN_COLAB:
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:
        from google.colab import files
        print("No se encontró la carpeta src/. Suba el .zip del proyecto (aca_2_machine_learning.zip):")
        subidos = files.upload()
        for nombre in subidos:
            if nombre.endswith(".zip"):
                import zipfile; zipfile.ZipFile(nombre).extractall("/content")
    ROOT = _buscar_raiz()
assert ROOT is not None, ("No se encontró la carpeta src/. Local: ejecute el cuaderno desde la raíz del repositorio (o notebooks/). "
                          "Colab: suba el .zip del proyecto cuando se lo pida, o ponga REPO_URL.")
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("Raíz del proyecto:", ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from src import data_dictionary as dd
from src import db
from src.preprocessing import FULL, EARLY
from src.train import split_data, RANDOM_STATE
from src.evaluate import compute_metrics
from src.nn.data import TabularEncoder
from src.nn.models import build_model, count_parameters, ARCH_NAMES
from src.nn import training as T

SEED = RANDOM_STATE            # 42, igual que en el ACA 1
LIMPIEZA_COMPLETA = True   # True = muestra TODO el proceso de limpieza del ACA 1 paso a paso | False = solo un resumen/auditoría y usa directamente el dataset limpio
QUICK = False                  # True = prueba rápida (menos configuraciones y épocas)
MAX_EPOCHS, PATIENCE = (40, 8) if QUICK else (100, 15)
T.set_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50, "display.width", 160)
print(f"PyTorch {torch.__version__} | GPU disponible: {torch.cuda.is_available()} (se entrena en CPU: las redes son pequeñas)")

---
# 📊 PARTE 1 — Exploración y gestión de datos con **SQL** (Paso 1)

El enunciado pide gestionar el dataset con una base de datos SQL. Usamos **SQLite**: viene incluida en Python (`import sqlite3`), **no requiere cuenta, servidor ni pago**, y la base es un único archivo (`data/estudiantes.db`) que se versiona en el repositorio. La aplicación Streamlit lee los datos desde esta misma base.

## 1.1 Creación de la base e inserción de datos
`src/db.py` carga el CSV original de UCI en la tabla `estudiantes_raw`, aplica la limpieza determinista del ACA 1 (nombres normalizados, tipos y validaciones) y guarda el resultado en la tabla `estudiantes`. También crea la vista `resumen_target`.

In [ ]:
ruta_db = db.build_db()          # recrea data/estudiantes.db desde data/data.csv
print("Base SQLite:", ruta_db, f"({ruta_db.stat().st_size/1024:.0f} KB)")
db.query("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name")

## 1.2 Consultas SQL de exploración

In [ ]:
print("→ Distribución de la variable objetivo (vista resumen_target)")
display(db.query("SELECT * FROM resumen_target"))

print("→ Perfil académico promedio por clase")
display(db.query("""
    SELECT target,
           COUNT(*)                                          AS estudiantes,
           ROUND(AVG(curricular_units_1st_sem_approved), 2)  AS aprobadas_1er_sem,
           ROUND(AVG(curricular_units_2nd_sem_approved), 2)  AS aprobadas_2do_sem,
           ROUND(AVG(curricular_units_1st_sem_grade), 2)     AS nota_1er_sem,
           ROUND(100.0 * AVG(tuition_fees_up_to_date), 1)    AS pct_matricula_al_dia,
           ROUND(100.0 * AVG(scholarship_holder), 1)         AS pct_becarios
    FROM estudiantes GROUP BY target ORDER BY estudiantes DESC
"""))

print("→ Programas académicos con mayor tasa de deserción (mínimo 50 estudiantes)")
top = db.query("""
    SELECT course, COUNT(*) AS estudiantes,
           ROUND(100.0 * SUM(target = 'Dropout') / COUNT(*), 1) AS pct_desercion
    FROM estudiantes GROUP BY course HAVING COUNT(*) >= 50
    ORDER BY pct_desercion DESC LIMIT 8
""")
top.insert(1, "programa", top["course"].map(lambda c: dd.code_label("course", c)))
display(top)

## 1.3 Limpieza y preprocesamiento de los datos (heredado del ACA 1)
La limpieza es **la misma del ACA 1** y vive en `src/data_loader.py → clean_dataset`, que es la que construyó la tabla `estudiantes` de SQLite. El interruptor `LIMPIEZA_COMPLETA` (celda de configuración) decide cómo se presenta:
- **`True`** → se ejecuta y se explica **todo el proceso paso a paso** sobre una copia (`df_clean`): bitácora → target → duplicados → nulos → tipos → consistencia lógica → outliers → validaciones → comparación antes/después. Al final se **comprueba que el resultado coincide exactamente** con `clean_dataset`.
- **`False`** → solo se muestra un **resumen y auditoría** del dataset ya limpio (para ejecutar más rápido o si la limpieza ya se revisó).

El flujo es el de los talleres de limpieza: **auditar → duplicados → nulos → texto/tipos → consistencia → outliers → validar**.

In [ ]:

if not LIMPIEZA_COMPLETA:
    from src.data_loader import load_raw, clean_dataset, consistency_report
    raw = load_raw(); limpio = clean_dataset(raw, strict=True)
    print("LIMPIEZA_COMPLETA = False → resumen de la limpieza del ACA 1 (src/data_loader.py → clean_dataset):")
    print("  1) nombres de columna a snake_case (se quitan BOM y tabulador; Nacionality → nationality)")
    print("  2) tipos: nominales y binarias a entero; application_order a numérica")
    print("  3) se rechazan registros imposibles (reglas duras); las reglas blandas solo se marcan")
    print("  4) no se eliminan outliers legítimos (la winsorización se evaluó como opción en el ACA 1)\n")
    display(pd.DataFrame({
        "CSV original": [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum()), raw.columns[0], raw.columns[4]],
        "Después de la limpieza": [limpio.shape[0], limpio.shape[1], int(limpio.isna().sum().sum()), int(limpio.duplicated().sum()), limpio.columns[0], limpio.columns[4]],
    }, index=["filas", "columnas", "nulos", "duplicados", "nombre columna 1", "nombre columna 5"]))
    print("Reporte de consistencia (reglas duras = 0; las «blandas» son casos atípicos legítimos que se conservan):")
    print(json.dumps(consistency_report(limpio), indent=2, ensure_ascii=False, default=str)[:1500])


### 1.3.1 Carga del CSV original, normalización de nombres y clasificación de variables

In [ ]:
if LIMPIEZA_COMPLETA:
    from src.data_loader import load_raw, normalizar_nombre, clean_dataset, RAW_PATH

    df_raw = load_raw(RAW_PATH)
    print(f"df_raw -> {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas | primeras columnas originales: {list(df_raw.columns[:3])}")

    df_ini = df_raw.copy()
    df_ini.columns = [normalizar_nombre(c) for c in df_ini.columns]      # quita BOM/tab y pasa a snake_case
    TARGET = dd.TARGET
    BINARY_COLS, CAT_COLS, NUM_COLS = list(dd.BINARY_FEATURES), list(dd.NOMINAL_FEATURES), list(dd.NUMERIC_FEATURES)
    FEATURES = [c for c in df_ini.columns if c != TARGET]
    faltan = set(BINARY_COLS + CAT_COLS + NUM_COLS) ^ set(FEATURES)
    assert not faltan, f"Columnas sin clasificar o inexistentes: {faltan}"
    print(f"Binarias ({len(BINARY_COLS)}), categóricas codificadas ({len(CAT_COLS)}), numéricas ({len(NUM_COLS)}) | columnas ahora: {list(df_ini.columns[:3])}")

    # Copia de trabajo y bitácora (df_raw / df_ini quedan intactos para comparar)
    df_clean = df_ini.copy()
    bitacora = {"filas_iniciales": len(df_clean), "columnas_iniciales": df_clean.shape[1]}
    print(bitacora)

### 1.3.2 Estandarización del target y duplicados

In [ ]:
if LIMPIEZA_COMPLETA:
    # Quitar espacios y unificar formato del texto del target
    df_clean[TARGET] = df_clean[TARGET].astype(str).str.strip().str.title()
    # Un registro sin target no sirve para aprendizaje supervisado y no se puede imputar
    sin_target = df_clean[TARGET].isin(["Nan", "", "None"])
    bitacora["filas_sin_target_eliminadas"] = int(sin_target.sum())
    df_clean = df_clean.loc[~sin_target].copy()
    print("Categorías del target:", sorted(df_clean[TARGET].unique()), "| filas sin target eliminadas:", bitacora["filas_sin_target_eliminadas"])

    antes = len(df_clean)
    df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)
    bitacora["duplicados_eliminados"] = antes - len(df_clean)
    print(f"Duplicados eliminados: {bitacora['duplicados_eliminados']} | Filas restantes: {len(df_clean):,}")

### 1.3.3 Valores nulos y corrección de tipos
Estrategia: columna con **> 40 %** de nulos → se elimina; numéricas → **mediana** (robusta a sesgo y outliers); binarias/categóricas codificadas → **moda**. Alternativa multivariada: `KNNImputer` (comentada). Luego se corrigen los tipos (enteros para códigos y conteos), lo que además ahorra memoria.

In [ ]:
if LIMPIEZA_COMPLETA:
    UMBRAL_COLUMNA = 0.40
    pct_nulos = df_clean.isna().mean()
    cols_eliminar = pct_nulos[pct_nulos > UMBRAL_COLUMNA].index.tolist()
    if cols_eliminar:
        df_clean = df_clean.drop(columns=cols_eliminar)
    bitacora["columnas_eliminadas_por_nulos"] = cols_eliminar

    num_act = [c for c in NUM_COLS if c in df_clean.columns]
    cat_act = [c for c in BINARY_COLS + CAT_COLS if c in df_clean.columns]
    imputados = {}
    for c in df_clean.columns[df_clean.isna().any()]:
        n = int(df_clean[c].isna().sum())
        if c in num_act:
            df_clean[c] = df_clean[c].fillna(df_clean[c].median()); imputados[c] = (n, "mediana")
        elif c in cat_act:
            df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0]); imputados[c] = (n, "moda")
    bitacora["nulos_imputados"] = imputados
    print("Columnas eliminadas por exceso de nulos:", cols_eliminar or "ninguna")
    print("Imputaciones realizadas:", imputados or "ninguna (no había nulos)", "| nulos restantes:", int(df_clean.isna().sum().sum()))
    # --- Alternativa multivariada ---
    # from sklearn.impute import KNNImputer
    # df_clean[num_act] = KNNImputer(n_neighbors=5).fit_transform(df_clean[num_act])

    # Tipos de dato
    mem_antes = df_clean.memory_usage(deep=True).sum() / 1024
    for c in BINARY_COLS: df_clean[c] = df_clean[c].astype("int8")
    for c in CAT_COLS:    df_clean[c] = df_clean[c].astype("int16")
    for c in num_act:
        if (df_clean[c] % 1 == 0).all():
            df_clean[c] = pd.to_numeric(df_clean[c], downcast="integer")      # conteos y edad -> enteros
    print(f"Memoria: {mem_antes:,.1f} KB -> {df_clean.memory_usage(deep=True).sum() / 1024:,.1f} KB")

### 1.3.4 Consistencia lógica
- **Reglas duras** → registros imposibles: se **eliminan**.
- **Reglas blandas** → casos atípicos pero posibles (p. ej. un desertor con 0 materias inscritas): **no se borran**, se **marcan** en `flag_inconsistencia` para poder auditarlos.

In [ ]:
if LIMPIEZA_COMPLETA:
    def reglas_consistencia(d):
        s1, s2 = "curricular_units_1st_sem_", "curricular_units_2nd_sem_"
        duras = {
            "admission_grade fuera de [0, 200]": ~d["admission_grade"].between(0, 200),
            "previous_qualification_grade fuera de [0, 200]": ~d["previous_qualification_grade"].between(0, 200),
            "nota 1er sem fuera de [0, 20]": ~d[s1 + "grade"].between(0, 20),
            "nota 2do sem fuera de [0, 20]": ~d[s2 + "grade"].between(0, 20),
            "edad al matricular < 14": d["age_at_enrollment"] < 14,
            "conteos curriculares negativos": (d[[c for c in d.columns if c.startswith("curricular_units") and "grade" not in c]] < 0).any(axis=1),
        }
        blandas = {
            "1er sem: aprobadas > inscritas": d[s1 + "approved"] > d[s1 + "enrolled"],
            "2do sem: aprobadas > inscritas": d[s2 + "approved"] > d[s2 + "enrolled"],
            "1er sem: nota > 0 sin materias inscritas": (d[s1 + "grade"] > 0) & (d[s1 + "enrolled"] == 0),
            "2do sem: nota > 0 sin materias inscritas": (d[s2 + "grade"] > 0) & (d[s2 + "enrolled"] == 0),
            "1er sem: inscritas > 0 y nota == 0": (d[s1 + "enrolled"] > 0) & (d[s1 + "grade"] == 0),
            "2do sem: inscritas > 0 y nota == 0": (d[s2 + "enrolled"] > 0) & (d[s2 + "grade"] == 0),
            "sin materias inscritas en ambos semestres": (d[s1 + "enrolled"] == 0) & (d[s2 + "enrolled"] == 0),
        }
        return duras, blandas

    duras, blandas = reglas_consistencia(df_clean)
    display(pd.DataFrame([{"tipo": "dura", "regla": k, "filas": int(v.sum())} for k, v in duras.items()] +
                         [{"tipo": "blanda", "regla": k, "filas": int(v.sum())} for k, v in blandas.items()]))

    invalidos = pd.concat(duras.values(), axis=1).any(axis=1)
    bitacora["filas_invalidas_eliminadas"] = int(invalidos.sum())
    df_clean = df_clean.loc[~invalidos].reset_index(drop=True)
    _, blandas = reglas_consistencia(df_clean)
    df_clean["flag_inconsistencia"] = pd.concat(blandas.values(), axis=1).any(axis=1).astype("int8")
    bitacora["filas_marcadas_sospechosas"] = int(df_clean["flag_inconsistencia"].sum())
    print(f"Filas inválidas eliminadas: {bitacora['filas_invalidas_eliminadas']} | marcadas como sospechosas: {bitacora['filas_marcadas_sospechosas']} ({df_clean['flag_inconsistencia'].mean() * 100:.1f} %)")
    display(df_clean.groupby(TARGET)["flag_inconsistencia"].mean().mul(100).round(2).to_frame("% sospechosas por clase"))

### 1.3.5 Outliers: IQR y winsorización
- Todos los valores están dentro de los rangos documentados por UCI, así que **no son errores de captura**.
- Recortar la edad afectaría a ≈ 10 % de los estudiantes (mayores de ~34 años), que además tienen la mayor tasa de deserción: su edad real es información útil.
- Por eso **no se recortan los datos**. En el ACA 1 la winsorización se evaluó como opción en la Regresión Logística. Para las redes se usa la estandarización con **recorte de z-scores a ±5** dentro del codificador (amortigua extremos sin borrar filas).

In [ ]:
if LIMPIEZA_COMPLETA:
    from src.preprocessing import WinsorizadorIQR
    COLS_WINSOR = ["previous_qualification_grade", "admission_grade", "age_at_enrollment"]
    w = WinsorizadorIQR(COLS_WINSOR, k=1.5).fit(df_clean)     # solo ilustrativo
    display(pd.DataFrame({c: {"lim_inf": li, "lim_sup": ls, "n_afectados": int(((df_clean[c] < li) | (df_clean[c] > ls)).sum())}
                          for c, (li, ls) in w.limites_.items()}).T)
    edad_alta = df_clean["age_at_enrollment"] > w.limites_["age_at_enrollment"][1]
    print(f"Estudiantes que se recortarían por edad: {int(edad_alta.sum())} | su tasa de deserción: "
          f"{(df_clean.loc[edad_alta, TARGET] == 'Dropout').mean() * 100:.1f} % (vs. {(df_clean.loc[~edad_alta, TARGET] == 'Dropout').mean() * 100:.1f} % del resto)")
    df_wins = w.transform(df_clean)
    fig, axes = plt.subplots(2, 3, figsize=(17, 7))
    for j, c in enumerate(COLS_WINSOR):
        sns.boxplot(x=df_clean[c], ax=axes[0, j], color="lightcoral"); axes[0, j].set_title(f"ORIGINAL · {c}", fontsize=10)
        sns.boxplot(x=df_wins[c], ax=axes[1, j], color="mediumseagreen"); axes[1, j].set_title(f"SI SE WINSORIZARA · {c}", fontsize=10)
        axes[0, j].set_xlabel(""); axes[1, j].set_xlabel("")
    plt.tight_layout(); plt.show()
    bitacora["winsorizacion"] = "no se aplica a los datos; las redes usan z-score con recorte a ±5"

### 1.3.6 Validación de calidad, coherencia con el pipeline y comparación antes → después

In [ ]:
if LIMPIEZA_COMPLETA:
    MAPA_TARGET = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
    assert df_clean.isna().sum().sum() == 0, "Persisten nulos"
    assert df_clean.drop(columns=["flag_inconsistencia"]).duplicated().sum() == 0, "Persisten duplicados"
    assert df_clean["admission_grade"].between(0, 200).all() and df_clean["previous_qualification_grade"].between(0, 200).all()
    assert df_clean["curricular_units_1st_sem_grade"].between(0, 20).all() and df_clean["curricular_units_2nd_sem_grade"].between(0, 20).all()
    assert (df_clean["age_at_enrollment"] >= 14).all(), "edad inválida"
    assert set(df_clean[TARGET].unique()) == set(MAPA_TARGET), "categorías de target inesperadas"

    # Garantía de coherencia: esta limpieza paso a paso produce EXACTAMENTE el mismo dataset que el pipeline (src/data_loader.clean_dataset)
    ref = clean_dataset(df_raw)
    pd.testing.assert_frame_equal(df_clean.drop(columns=["flag_inconsistencia"]).reset_index(drop=True), ref.reset_index(drop=True), check_dtype=False)
    print("✅ Validaciones de calidad superadas y el dataset coincide con el del pipeline (clean_dataset).\n")

    display(pd.DataFrame({
        "antes (df_raw)": [len(df_raw), df_raw.shape[1], int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum())],
        "después (df_clean)": [len(df_clean), df_clean.shape[1] - 1, int(df_clean.isna().sum().sum()), int(df_clean.drop(columns=["flag_inconsistencia"]).duplicated().sum())],
    }, index=["filas", "columnas", "nulos", "duplicados"]))
    display(pd.concat([df_ini[TARGET].value_counts(normalize=True).mul(100).rename("target antes %"),
                       df_clean[TARGET].value_counts(normalize=True).mul(100).rename("target después %")], axis=1).round(2))
    print("Bitácora de limpieza:"); print(json.dumps(bitacora, indent=2, ensure_ascii=False, default=str))

**Resumen de decisiones de limpieza**

| Aspecto | Decisión | Justificación |
|---|---|---|
| Nombres | `snake_case`, `Nacionality → nationality` | Evitar errores por espacios/tab ocultos |
| Duplicados | Eliminar exactos | Evitan sobre-representar casos (había 0) |
| Nulos | Mediana (num) / moda (cat); columna > 40 % se elimina | Robusto a sesgo; el dataset original no tiene nulos |
| Tipos | Enteros para códigos y conteos | Tipos coherentes |
| Consistencia | Duras → eliminar; blandas → `flag_inconsistencia` | No perder desertores reales con 0 materias |
| Outliers | Se conservan; z-score recortado a ±5 en el codificador de las redes | Son casos legítimos; recortar la edad borra información del grupo de mayor riesgo |
| Target | Texto (`Dropout`/`Enrolled`/`Graduate`), multiclase | La matriz de confusión y la app muestran los nombres de clase |

## 1.4 Carga del DataFrame desde SQL y EDA breve
El EDA completo (nulos, duplicados, outliers, correlaciones, consistencia lógica) está en el cuaderno del ACA 1 (`notebooks/aca1/`). Aquí solo se confirma que los datos leídos desde SQL son los mismos y se resumen los hallazgos que justifican las decisiones de modelado.

In [ ]:
df = db.load_clean_from_db()
print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas | nulos: {int(df.isna().sum().sum())} | duplicados: {int(df.duplicated().sum())}")

# Los datos de SQL coinciden con el CSV limpio del ACA 1
from src.data_loader import load_clean
assert df.equals(load_clean()), "La tabla SQL difiere del CSV limpio"
print("✔ La tabla `estudiantes` coincide exactamente con el CSV limpio del ACA 1")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
orden = dd.CLASSES
sns.countplot(data=df, x="target", order=orden, palette=dd.CLASS_COLORS, ax=ax[0]); ax[0].set_title("Clases (desbalance)")
sns.boxplot(data=df, x="target", y="curricular_units_1st_sem_approved", order=orden, palette=dd.CLASS_COLORS, ax=ax[1]); ax[1].set_title("Unidades aprobadas 1.er semestre")
sns.boxplot(data=df, x="target", y="curricular_units_2nd_sem_grade", order=orden, palette=dd.CLASS_COLORS, ax=ax[2]); ax[2].set_title("Nota promedio 2.º semestre")
plt.tight_layout(); plt.show()

**Hallazgos que condicionan el modelado neuronal**
- **Desbalance moderado** (Graduado ≈ 50 %, Deserción ≈ 32 %, Matriculado ≈ 18 %) → pérdida con **pesos de clase** y métrica **macro-F1** (no solo accuracy).
- **Variables nominales de alta cardinalidad** (programa, ocupación de los padres, calificación previa: 17–46 códigos) → conviene usar **capas de *embeddings*** en lugar de One-Hot disperso.
- **Escalas muy distintas** (notas 0–200, unidades 0–20, tasas 0–1) → las redes exigen **estandarizar** las numéricas.
- Pocos datos (≈ 4.400 filas) → riesgo de sobreajuste → **dropout, *weight decay* y *early stopping***.

---
# 🧹 PARTE 2 — Preparación y pipeline para redes neuronales (Paso 2)

Se conserva la **limpieza del ACA 1** (ya aplicada en la tabla `estudiantes`) y la **misma partición estratificada 80/20** (`random_state=42`), de modo que los resultados sean comparables. Se evalúan dos escenarios, como en el ACA 1:
- **Completo**: toda la información académica (1.er y 2.º semestre).
- **Alerta temprana**: solo información disponible al cierre del 1.er semestre (sin variables del 2.º semestre).

## 2.1 Codificador tabular (`src/nn/data.py`)
`TabularEncoder` convierte cada fila en dos tensores, ajustándose **solo con datos de entrenamiento** (sin fuga):
- `x_num`: binarias + numéricas + tasas de aprobación por semestre, **estandarizadas** (z-score, recortadas a ±5).
- `x_cat`: índice de cada variable nominal (0 = categoría desconocida o infrecuente, < 10 apariciones) para las capas de *embeddings*.

In [ ]:
particiones = {}
for fs in (FULL, EARLY):
    X_train, X_test, y_train, y_test = split_data(df, fs)
    particiones[fs.name] = dict(fs=fs, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
    print(f"[{fs.name:8s}] entrenamiento: {len(X_train):,} | prueba: {len(X_test):,} | variables de entrada: {len(fs.raw_columns)}")

p = particiones["completo"]
print("\nDistribución por clase (estratificada):")
display(pd.DataFrame({"entrenamiento %": p["y_train"].value_counts(normalize=True).mul(100).round(1),
                      "prueba %": p["y_test"].value_counts(normalize=True).mul(100).round(1)}))

In [ ]:
enc = TabularEncoder(FULL).fit(p["X_train"])
xn, xc = enc.transform(p["X_train"])
print(f"x_num: {xn.shape} (float32) | x_cat: {xc.shape} (int64)")
print("Vocabulario por variable nominal (incluye el índice 0 = desconocido):")
print(dict(zip(enc.cat_cols, enc.cardinalities)))

# Verificación anti-fuga: las estadísticas vienen solo de entrenamiento
enc_todo = TabularEncoder(FULL).fit(df[FULL.raw_columns])
assert not np.allclose(enc.mean_, enc_todo.mean_)
print("\n✔ El codificador se ajusta solo con entrenamiento (medias distintas a las del dataset completo)")
print("✔ Las numéricas quedan centradas: media ≈", np.round(xn.mean(), 3), "| desv. ≈", np.round(xn.std(), 3))

## 2.2 Desbalance: pesos de clase
En lugar de generar datos sintéticos, la pérdida se pondera con `n / (k · nᶜ)` (equivalente a `class_weight='balanced'` del ACA 1): equivocarse en la clase minoritaria «Matriculado» cuesta más.

In [ ]:
y_int = T.encode_labels(p["y_train"])
pesos = T.class_weights(y_int).numpy()
display(pd.DataFrame({"clase": T.CLASSES, "peso en la pérdida": pesos.round(3)}).set_index("clase").T)

---
# 🏗️ PARTE 3 — Arquitecturas de redes neuronales (Paso 3)

## 3.1 ¿Qué arquitectura usar? (justificación)
| Arquitectura | ¿Aplica a este problema? |
|---|---|
| **CNN** (convolucional) | Diseñada para imágenes/señales con estructura espacial. Los datos son una **tabla** sin vecindad espacial → no aplica. |
| **RNN / LSTM / GRU** | Para **secuencias**. El dataset tiene una fila por estudiante con dos semestres resumidos, no una serie temporal larga → aporta poco. |
| **Modelos de Hugging Face** (BERT, etc.) | Preentrenados en **texto/imágenes**. No hay texto en este dataset; adaptarlos exigiría convertir filas en frases, sin ventaja clara. |
| **MLP con *embeddings*** ✅ | Es la red de referencia para datos tabulares y la que presentó el docente (`Sequential`, capas densas, ReLU, softmax). Las *embeddings* tratan bien las variables nominales. |
| **FT-Transformer** ✅ | **Transformer para tablas** (Gorishniy et al., 2021): cada variable es un *token* y la **auto-atención** modela interacciones entre variables. Es el equivalente tabular de la arquitectura que mueve los LLM. |

Se implementan **dos redes** y se elige la mejor por validación cruzada (macro-F1), comparándolas además con los modelos clásicos del ACA 1.

## 3.2 MLP con *embeddings*
```
 Numéricas (z-score) ───────────────────────────┐
 Nominales ─► Embedding(c_i → 4) por variable ──┴─► concat ─► Linear → ReLU → Dropout ─► … ─► Linear(3) ─► softmax
```
## 3.3 FT-Transformer
```
 Cada variable numérica x_j ─► token  x_j · w_j + b_j      ┐
 Cada variable nominal      ─► token  Embedding            ├─► [CLS] + tokens ─► Encoder Transformer (L capas,
 Token aprendible [CLS] ────────────────────────────────── ┘     auto-atención multi-cabeza + FFN, pre-LayerNorm)
                                                                  ─► salida del [CLS] ─► LayerNorm ─► Linear(3) ─► softmax
```

In [ ]:
enc = TabularEncoder(FULL).fit(p["X_train"])
demo = {
    "mlp":            {"emb_dim": 4, "hidden": [64], "dropout": 0.3},
    "ft_transformer": {"d_token": 32, "n_layers": 2, "n_heads": 4, "ffn_mult": 2, "dropout": 0.2},
}
for arch, prm in demo.items():
    m = build_model(arch, enc.n_num, enc.cardinalities, 3, prm)
    print(f"{'='*100}\n{ARCH_NAMES[arch]} — parámetros entrenables: {count_parameters(m):,}\n{'='*100}")
    print(m if arch == "mlp" else "\n".join(str(m).splitlines()[:14]) + "\n  ...")

---
# 🏋️ PARTE 4 — Entrenamiento (Paso 3)

`src/nn/training.py → fit_nn` implementa el ciclo de entrenamiento:
- **Pérdida:** entropía cruzada **ponderada por clase** (softmax + log-verosimilitud).
- **Optimizador:** **AdamW** (Adam con *weight decay* desacoplado como regularización).
- **Mini-lotes** de 128 filas; cada recorrido completo de los datos es una **época**.
- **Early stopping:** se monitorea el **macro-F1 de validación** (15 % del entrenamiento, estratificado) y se restauran los pesos de la mejor época; detiene tras `PATIENCE` épocas sin mejora. Es la defensa contra el *overfitting* (memorizar) que explicó el docente.
- **Curvas:** por época se guardan pérdida, accuracy y macro-F1 de entrenamiento y validación.

## 4.1 Demostración: una red, un entrenamiento, sus curvas

In [ ]:
def plot_history(h, titulo=""):
    ep = np.arange(1, len(h["val_f1"]) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    for a, (tr, va, nom) in zip(ax, [("train_loss", "val_loss", "Pérdida (entropía cruzada ponderada)"),
                                    ("train_acc", "val_acc", "Accuracy"), ("train_f1", "val_f1", "Macro-F1")]):
        a.plot(ep, h[tr], label="Entrenamiento", lw=2); a.plot(ep, h[va], label="Validación", lw=2)
        a.axvline(h["best_epoch"], ls=":", c="gray", label=f"época elegida ({h['best_epoch']})")
        a.set_title(nom); a.set_xlabel("Época"); a.legend()
    fig.suptitle(titulo, y=1.03, fontsize=13); plt.tight_layout(); plt.show()

cfg_demo = {"arch": demo["mlp"], "train": {"lr": 1e-3, "weight_decay": 1e-2}}
t0 = time.perf_counter()
clf_demo, h_demo = T.fit_with_inner_val("mlp", cfg_demo, FULL, p["X_train"], p["y_train"], max_epochs=MAX_EPOCHS, patience=PATIENCE)
print(f"Entrenado en {time.perf_counter()-t0:.1f} s | épocas ejecutadas: {h_demo['epochs_run']} | mejor época: {h_demo['best_epoch']}")
plot_history(h_demo, "MLP (configuración inicial) — escenario completo")

**Lectura de las curvas.** La pérdida de entrenamiento sigue bajando, pero la de validación se estanca (o sube) pasada cierta época: ahí empieza el sobreajuste. *Early stopping* se queda con la época de mejor macro-F1 de validación.

---
# 🔎 PARTE 5 — Selección de hiperparámetros por validación cruzada (Paso 3)

Igual que en el ACA 1, cada configuración se evalúa con **validación cruzada estratificada de 5 pliegues** sobre el conjunto de entrenamiento y se elige la de mayor **macro-F1**. En cada pliegue el codificador se vuelve a ajustar solo con los datos de entrenamiento del pliegue (sin fuga). El conjunto de prueba **no interviene**.

Rejilla (ver `src/nn/training.py → GRIDS`): 4 configuraciones de MLP (ancho/profundidad, dimensión de *embedding*, dropout, *weight decay*, tasa de aprendizaje) y 2 de FT-Transformer (dimensión de token, capas, cabezas, dropout).

In [ ]:
seleccion = {}      # seleccion[modalidad][arch] = dict(cfg, cv, tabla, tiempo)
for fs in (FULL, EARLY):
    P = particiones[fs.name]; seleccion[fs.name] = {}
    for arch in ("mlp", "ft_transformer"):
        grid = T.GRIDS[arch][:2] if QUICK else T.GRIDS[arch]
        t0 = time.perf_counter()
        cfg, cv, tabla = T.select_config(arch, fs, P["X_train"], P["y_train"], grid=grid, max_epochs=MAX_EPOCHS, patience=PATIENCE)
        seleccion[fs.name][arch] = dict(cfg=cfg, cv=cv, tabla=tabla, n_cfg=len(grid), tiempo=time.perf_counter() - t0)
        print(f"   → {ARCH_NAMES[arch]}: mejor macro-F1 CV = {cv['f1_macro']['media']:.4f} ± {cv['f1_macro']['desv']:.4f}  ({seleccion[fs.name][arch]['tiempo']:.0f} s)\n")

In [ ]:
for mod in seleccion:
    for arch in seleccion[mod]:
        print(f"── {mod} · {ARCH_NAMES[arch]}")
        display(seleccion[mod][arch]["tabla"].round(4))

---
# 📈 PARTE 6 — Entrenamiento final y evaluación en prueba (Paso 4)

Con la mejor configuración de cada arquitectura se entrena sobre **todo el 80 % de entrenamiento** (con 15 % interno para *early stopping*) y se evalúa **una sola vez** en el 20 % de prueba: accuracy, balanced accuracy, macro-F1, precisión/recall/F1 por clase, ROC-AUC y matriz de confusión.

In [ ]:
resultados = {}    # resultados[modalidad][arch] = {...}  (formato compatible con la app)
clasificadores = {}
for fs in (FULL, EARLY):
    P = particiones[fs.name]; resultados[fs.name] = {}; clasificadores[fs.name] = {}
    for arch in ("mlp", "ft_transformer"):
        s = seleccion[fs.name][arch]
        t0 = time.perf_counter()
        clf, hist = T.fit_with_inner_val(arch, s["cfg"], fs, P["X_train"], P["y_train"], max_epochs=MAX_EPOCHS, patience=PATIENCE)
        prueba = T.evaluate_on_test(clf, P["X_test"], P["y_test"])
        clasificadores[fs.name][arch] = clf
        resultados[fs.name][arch] = {
            "nombre": ARCH_NAMES[arch],
            "mejores_hiperparametros": {**s["cfg"]["arch"], **{k: v for k, v in clf.train_params.items()},
                                        "class_weight": "balanced (en la pérdida)", "optimizador": "AdamW",
                                        "early_stopping": f"macro-F1 de validación, paciencia {PATIENCE}"},
            "combinaciones_evaluadas": s["n_cfg"],
            "cv": s["cv"],
            "prueba": prueba,
            "historial": T.history_to_json(hist),
            "parametros": T.n_parameters(clf),
            "tiempo_busqueda_s": round(s["tiempo"], 1),
        }
        print(f"[{fs.name:8s}] {ARCH_NAMES[arch]:34s} CV macro-F1={s['cv']['f1_macro']['media']:.4f} | "
              f"PRUEBA: acc={prueba['accuracy']:.4f}  macro-F1={prueba['f1_macro']:.4f}  bal.acc={prueba['balanced_accuracy']:.4f}  AUC={prueba['roc_auc_ovr_macro']:.4f}  "
              f"(época {hist['best_epoch']}/{hist['epochs_run']}, {time.perf_counter()-t0:.0f} s)")

ganador = {mod: max(resultados[mod], key=lambda a: resultados[mod][a]["cv"]["f1_macro"]["media"]) for mod in resultados}
print("\nRed seleccionada por macro-F1 en validación cruzada (no en prueba):", {m: ARCH_NAMES[a] for m, a in ganador.items()})

## 6.1 Curvas de aprendizaje de las redes finales

In [ ]:
for mod in resultados:
    for arch in resultados[mod]:
        plot_history(resultados[mod][arch]["historial"], f"{ARCH_NAMES[arch]} — escenario {mod}")

## 6.2 Matrices de confusión y reporte por clase (conjunto de prueba)

In [ ]:
etq = [dd.CLASS_LABELS_ES[c] for c in T.CLASSES]
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for i, mod in enumerate(resultados):
    for j, arch in enumerate(resultados[mod]):
        cm = np.array(resultados[mod][arch]["prueba"]["matriz_confusion"]["valores"])
        ConfusionMatrixDisplay(cm, display_labels=etq).plot(ax=axes[i, j], cmap="Blues", colorbar=False, values_format="d")
        axes[i, j].set_title(f"{ARCH_NAMES[arch]}\nescenario {mod} · macro-F1 {resultados[mod][arch]['prueba']['f1_macro']:.3f}", fontsize=10)
plt.tight_layout(); plt.show()

for mod in resultados:
    a = ganador[mod]; P = particiones[mod]
    print(f"\n══ Reporte por clase — {ARCH_NAMES[a]} · escenario {mod} (prueba)")
    print(classification_report(P["y_test"], clasificadores[mod][a].predict(P["X_test"]), target_names=etq, digits=3))

---
# ⚖️ PARTE 7 — Comparación con los modelos clásicos del ACA 1 e interpretación (Paso 4)

Los modelos clásicos (Regresión Logística, Random Forest, HistGradientBoosting) se evaluaron en el ACA 1 con **la misma partición y la misma validación cruzada**, por lo que sus métricas (guardadas en `models/aca1/`) son directamente comparables.

In [ ]:
base = {}
for mod, f in (("completo", "metricas.json"), ("temprano", "metricas_temprano.json")):
    m = json.loads((ROOT / "models" / "aca1" / f).read_text(encoding="utf-8"))
    base[mod] = {k: m["modelos"][k] for k in ("logreg", "rf", "hgb")}

filas = []
for mod in resultados:
    for k, v in {**resultados[mod], **base[mod]}.items():
        filas.append({"escenario": mod, "modelo": v["nombre"].split(" (")[0], "tipo": "Red neuronal" if k in resultados[mod] else "Clásico (ACA 1)",
                      "macro-F1 CV": v["cv"]["f1_macro"]["media"], "± CV": v["cv"]["f1_macro"]["desv"],
                      "accuracy prueba": v["prueba"]["accuracy"], "macro-F1 prueba": v["prueba"]["f1_macro"],
                      "bal. acc. prueba": v["prueba"]["balanced_accuracy"], "recall Deserción": v["prueba"]["por_clase"]["Dropout"]["recall"],
                      "AUC OvR": v["prueba"].get("roc_auc_ovr_macro")})
comparacion = pd.DataFrame(filas)
display(comparacion.sort_values(["escenario", "macro-F1 CV"], ascending=[True, False]).round(4).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for a, mod in zip(ax, ["completo", "temprano"]):
    d = comparacion[comparacion.escenario == mod].sort_values("macro-F1 CV")
    colores = ["#13315C" if t == "Red neuronal" else "#9AA5B8" for t in d["tipo"]]
    a.barh(d["modelo"], d["macro-F1 CV"], xerr=d["± CV"], color=colores)
    a.scatter(d["macro-F1 prueba"], d["modelo"], c="#eb6834", zorder=3, label="macro-F1 en prueba")
    a.set_title(f"Escenario {mod}"); a.set_xlabel("macro-F1 (barras = CV ± desv.)"); a.set_xlim(0.55, 0.8); a.legend(loc="lower right")
plt.suptitle("Redes neuronales (azul oscuro) vs. modelos clásicos del ACA 1 (gris)", y=1.02); plt.tight_layout(); plt.show()

## 7.1 Importancia por permutación de la red seleccionada

In [ ]:
from src.nn.explain import permutation_importances
imps = {}
for mod in resultados:
    a = ganador[mod]; P = particiones[mod]
    imps[mod] = permutation_importances(clasificadores[mod][a], P["X_test"], P["y_test"], n_repeats=10, random_state=SEED, top=12)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for a_, mod in zip(ax, imps):
    d = pd.DataFrame(imps[mod]).iloc[::-1]
    a_.barh(d["etiqueta"], d["importancia_media"], xerr=d["importancia_desv"], color="#13315C")
    a_.set_title(f"{ARCH_NAMES[ganador[mod]]} — {mod}"); a_.set_xlabel("Caída media de macro-F1 al barajar la variable")
plt.tight_layout(); plt.show()

---
# 💾 PARTE 8 — Exportación para Streamlit (Paso 5)

Se guarda la red seleccionada de cada escenario en `models/` (`.pt` con pesos + codificador, y JSON con metadatos y métricas, incluidas las curvas). El `.pt` contiene solo tensores y tipos primitivos y se carga con `torch.load(weights_only=True)`.

In [ ]:
for fs in (FULL, EARLY):
    mod = fs.name; P = particiones[mod]; a = ganador[mod]
    metricas = T.export_modality(fs, a, clasificadores[mod][a], resultados[mod], base[mod],
                                 P["X_train"], P["X_test"], P["y_train"], P["y_test"], df, ROOT / "models")
    print(f"✔ {mod}: red desplegada = {ARCH_NAMES[a]} | ranking CV: {metricas['ranking_cv']}")

print()
for f in sorted((ROOT / "models").glob("*")):
    if f.is_file(): print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

## 8.1 Verificación: la app reproduce exactamente el cuaderno

In [ ]:
from src.predict import load_model, predict_one, predict_frame, check_artifacts
assert not any(check_artifacts().values()), check_artifacts()

for mod in ("completo", "temprano"):
    P = particiones[mod]; a = ganador[mod]
    reloaded = load_model(mod)
    X = P["X_test"]
    assert np.allclose(reloaded.predict_proba(X), clasificadores[mod][a].predict_proba(X), atol=1e-6)
    acc = (reloaded.predict(X) == P["y_test"]).mean()
    assert abs(acc - resultados[mod][a]["prueba"]["accuracy"]) < 1e-9
    print(f"✔ {mod}: el modelo recargado desde models/*.pt da las mismas probabilidades y accuracy de prueba = {acc:.4f}")

ej = particiones["completo"]["X_test"].iloc[0].to_dict()
r = predict_one(ej, "completo")
print("\nEjemplo (misma ruta que usa la app):", r.clase_es, {dd.CLASS_LABELS_ES[k]: round(v, 3) for k, v in r.probabilidades.items()})
print("Real:", dd.CLASS_LABELS_ES[particiones["completo"]["y_test"].iloc[0]])

In [ ]:
# Pruebas automáticas del repositorio (incluye el arranque de las 4 páginas de Streamlit)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(res.stdout[-1500:] or res.stderr[-1500:])

## 8.2 Descarga de los modelos (Colab)
Descargue `models_aca2.zip`, descomprímalo y copie su contenido en la carpeta `models/` del repositorio (junto a `models/aca1/`).

In [ ]:
# En Colab: descargar la carpeta models/ (modelos entrenados + métricas) para llevarla al repositorio
if IN_COLAB:
    import shutil
    shutil.make_archive("/content/models_aca2", "zip", ROOT / "models")
    from google.colab import files
    files.download("/content/models_aca2.zip")
else:
    print("Ejecución local: los modelos ya quedaron en", ROOT / "models")

---
# ✅ Conclusiones

In [ ]:
from IPython.display import Markdown
def fila(mod):
    a = ganador[mod]; r = resultados[mod][a]
    bmod = max(base[mod], key=lambda k: base[mod][k]["cv"]["f1_macro"]["media"]); b = base[mod][bmod]
    return a, r, bmod, b
txt = []
for mod, nombre in (("completo", "información completa"), ("temprano", "alerta temprana (1.er semestre)")):
    a, r, bmod, b = fila(mod)
    d = r["cv"]["f1_macro"]["media"] - b["cv"]["f1_macro"]["media"]
    txt.append(f"- **Escenario {nombre}.** La red seleccionada fue **{ARCH_NAMES[a]}** ({r['parametros']:,} parámetros): macro-F1 CV = {r['cv']['f1_macro']['media']:.3f} ± {r['cv']['f1_macro']['desv']:.3f}; "
               f"en prueba, accuracy = {r['prueba']['accuracy']:.3f}, macro-F1 = {r['prueba']['f1_macro']:.3f}, recall de deserción = {r['prueba']['por_clase']['Dropout']['recall']:.3f}. "
               f"Frente al mejor modelo clásico del ACA 1 ({b['nombre'].split(' (')[0]}, macro-F1 CV = {b['cv']['f1_macro']['media']:.3f}) la diferencia es de {d:+.3f}, "
               f"{'dentro' if abs(d) < max(r['cv']['f1_macro']['desv'], b['cv']['f1_macro']['desv']) else 'fuera'} de la variación entre pliegues.")
top3 = ", ".join(i["etiqueta"] for i in imps["completo"][:3])
txt.append(f"- **Variables más influyentes** (permutación, escenario completo): {top3}.")
Markdown("\n".join(txt).replace(",", ",") )

**Interpretación y límites (para el póster y la sustentación)**
1. **Las redes neuronales alcanzan un desempeño comparable al de los modelos clásicos, no superior.** Es un resultado esperable con ≈ 4.400 filas tabulares: los árboles con *boosting* son muy competitivos en este régimen. El valor del ACA 2 está en la arquitectura, el control del sobreajuste y las curvas de aprendizaje, no en un salto de métrica.
2. **El FT-Transformer** modela interacciones entre variables por auto-atención, pero con pocos datos no tiene ventaja clara sobre el MLP y cuesta más entrenarlo; por eso la selección se hace por validación cruzada y no por preferencia.
3. **La clase «Matriculado»** (minoritaria y ambigua) sigue siendo la más difícil, igual que en el ACA 1; la ponderación de clases mejora su recall a costa de algo de precisión.
4. **Caja negra.** A diferencia de un árbol de decisión, la red no explica su decisión; la importancia por permutación solo indica qué variables pesan, sin implicar causalidad.
5. **Ética.** Las probabilidades no son una medición validada de riesgo, los datos son de una sola institución portuguesa y las variables sensibles (sexo, nacionalidad, estado civil) pueden introducir sesgos: la herramienta es de apoyo académico, no de decisión automática.

## Siguientes pasos
1. Probar la app: `streamlit run app.py`
2. Publicarla en Streamlit Community Cloud y completar los enlaces en `src/project_info.py`.
3. Generar el póster: `python scripts/build_poster.py` → `docs/poster/poster.html` → Imprimir → Guardar como PDF.
4. Grabar el video de sustentación (3–5 min) siguiendo `docs/video/guion_sustentacion.md`.

## Referencias
Ver `src/project_info.py → REFERENCIAS_APA` (Gorishniy et al., 2021; Loshchilov & Hutter, 2019; Paszke et al., 2019; Realinho et al., 2021, 2022; Vaswani et al., 2017).